In [0]:
from pyspark.sql.functions import col, when, split, length

# 1. Read the cleansed data from the Silver table
df_silver_full = spark.table("workspace.my_database.customers_silver")

# 2. Implement streamlined transformation functions
df_gold_enriched = df_silver_full \
    .withColumn(
        # Logic 1: Since nulls are dropped in Silver, we just verify they aren't empty strings
        "has_complete_contact",
        when((col("email") != "") & (col("phone") != ""), True).otherwise(False)
    ) \
    .withColumn(
        # Logic 2: Direct split (No need to check for 'Not Provided' anymore)
        "email_provider",
        split(col("email"), "@").getItem(1)
    ) \
    .withColumn(
        # Logic 3: Direct length check
        "is_valid_phone_length",
        when(length(col("phone")) >= 10, True).otherwise(False)
    ) \
    .withColumn(
        # Logic 4: Age categorization remains exactly the same
        "age_group",
        when(col("age").isNull(), "Unknown")
        .when(col("age") < 18, "Minor")
        .when((col("age") >= 18) & (col("age") <= 35), "Young Adult")
        .when((col("age") > 35) & (col("age") <= 55), "Adult")
        .otherwise("Senior")
    )

# Save the enriched dataset as the new Gold table
df_gold_enriched.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.my_database.customers_gold")

print("Gold Layer has been enriched with optimized PySpark functions and saved successfully!")
# 4. Display the data to verify the new columns
display(df_gold_enriched)